<a href="https://colab.research.google.com/github/ReshmiMaulik/Causal-inference-code/blob/main/Claude_brackets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install git+https://github.com/microsoft/dowhy.git
import dowhy
from dowhy import CausalModel

import numpy as np
import pandas as pd

  Cloning https://github.com/microsoft/dowhy.git to /tmp/pip-req-build-hl7vmqmr
  Running command git clone --filter=blob:none --quiet https://github.com/microsoft/dowhy.git /tmp/pip-req-build-hl7vmqmr
  Resolved https://github.com/microsoft/dowhy.git to commit 1d1efe77b092661252038baad72dc5d53e35ebfa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.9/245.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 73.0 MB/s eta 0:00:00
  Created wheel for dowhy: filename=dowhy-0.0.0-py3-none-any.whl size=426797 sha256=a2e9043663a4ab4f83756cc5a713be392405e0e5d25604737e4b0da8ccf87303
  Stored in directory: /tmp/pip-ephem-wheel-cache-96exp_oe/wheels/4e/c8/a7/072344a2433ab157ed284bad

In [7]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from dowhy import CausalModel

# Install dowhy if not already installed
#!pip install dowhy

#DATA_PATH = "vscode__1_.csv"
url="https://raw.githubusercontent.com/sunnysong14/ContinualPerformanceValidityTSE2022/main/data/vscode.csv"

df = pd.read_csv(url, sep=",")
DATA_PATH = url
df1= df[['ndev','nf','ns','entrophy','sexp','rexp','exp','days_to_first_fix','nuc']].copy()
df1.rename(columns={'entrophy': 'entropy'}, inplace=True)
PROJECT_NAME = "VSCode"
OUTCOME = "days_to_first_fix"
TREATMENTS = ["ndev", "entropy", "sexp", "rexp"]
ADJUSTMENT_SET = []  # empty for all four treatments, per Section 2.3


def raw_linear_ate(df_arg, treatment, outcome=OUTCOME):
    """Reproduces the Tables 6-9 specification: raw-scale linear ATE via DoWhy."""
    model = CausalModel(data=df_arg, treatment=treatment, outcome=outcome,
                         common_causes=None)
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    est = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                 target_units="ate")
    ols = smf.ols(f"{outcome} ~ {treatment}", data=df_arg).fit()
    return est.value, ols.pvalues[treatment]


def log_transformed_ate(df_arg, treatment, outcome=OUTCOME):
    """
    Re-estimates the ATE using a log transform of the treatment in place
    of the raw treatment. Uses log1p(x) for treatments that are
    non-negative in this dataset (ndev, entropy, sexp). For rexp, which
    contains a small number of negative values in the raw VSCode data
    (210 of 51,846 rows, min = -17,128) despite being described in the
    manuscript as a non-negative "recent experience level" (Section
    2.2), we use the signed-log transform sign(x) * log1p(|x|), which
    compresses the right tail symmetrically for positive and negative
    values without requiring the negative rows to be dropped or
    clipped. This choice, and the presence of negative rexp values
    itself, should be reported alongside this sensitivity check.
    """
    df_local = df_arg.copy()
    log_col = f"log1p_{treatment}"
    if (df_local[treatment] < 0).any():
        df_local[log_col] = np.sign(df_local[treatment]) * np.log1p(df_local[treatment].abs())
    else:
        df_local[log_col] = np.log1p(df_local[treatment])
    model = CausalModel(data=df_local, treatment=log_col, outcome=outcome,
                         common_causes=None)
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    est = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                 target_units="ate")
    ols = smf.ols(f"{outcome} ~ {log_col}", data=df_local).fit()
    return est.value, ols.pvalues[log_col]


def standardized_ate(df_arg, treatment, outcome=OUTCOME):
    """
    Re-estimates the ATE using the treatment standardized to zero mean,
    unit variance. Coefficient is directly interpretable as the expected
    change in outcome (days) per one-standard-deviation increase in the
    treatment -- puts ndev, entropy, sexp, and rexp (which live on very
    different raw scales, Tables 2-5) on a common footing for comparing
    relative effect sizes within and across projects.
    """
    df_local = df_arg.copy()
    z_col = f"z_{treatment}"
    df_local[z_col] = (df_local[treatment] - df_local[treatment].mean()) / df_local[treatment].std()
    model = CausalModel(data=df_local, treatment=z_col, outcome=outcome,
                         common_causes=None)
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    est = model.estimate_effect(estimand, method_name="backdoor.linear_regression",
                                 target_units="ate")
    ols = smf.ols(f"{outcome} ~ {z_col}", data=df_local).fit()
    return est.value, ols.pvalues[z_col]


def quantile_contrast(df_arg, treatment, outcome=OUTCOME, q_low=0.25, q_high=0.75):
    """Q25->Q75 contrast via DoWhy's control_value/treatment_value, linear model held fixed."""
    model = CausalModel(data=df_arg, treatment=treatment, outcome=outcome,
                         common_causes=None)
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    q_lo, q_hi = df_arg[treatment].quantile(q_low), df_arg[treatment].quantile(q_high)
    contrast = model.estimate_effect(
        estimand, method_name="backdoor.linear_regression",
        control_value=q_lo, treatment_value=q_hi, target_units="ate",
    )
    return q_lo, q_hi, contrast.value


if __name__ == "__main__":
    processed_df = df1[TREATMENTS + [OUTCOME]].dropna()
    print(f"Loaded {PROJECT_NAME}: {len(processed_df)} rows\n")

    rows = []
    for t in TREATMENTS:
        raw_ate, raw_p = raw_linear_ate(processed_df, t)
        log_ate, log_p = log_transformed_ate(processed_df, t)
        z_ate, z_p = standardized_ate(processed_df, t)
        q_lo, q_hi, contrast = quantile_contrast(processed_df, t)

        rows.append({
            "treatment": t,
            "raw_ATE": raw_ate, "raw_p": raw_p,
            "log1p_ATE": log_ate, "log1p_p": log_p,
            "standardized_ATE_per_SD": z_ate, "standardized_p": z_p,
            "Q25": q_lo, "Q75": q_hi, "Q25_to_Q75_contrast_days": contrast,
            "sign_consistent_raw_vs_log": np.sign(raw_ate) == np.sign(log_ate),
            "sig_at_05_raw": raw_p < 0.05,
            "sig_at_05_log": log_p < 0.05,
            "sig_at_05_standardized": z_p < 0.05,
        })

    result = pd.DataFrame(rows)
    pd.set_option("display.width", 160)
    print(result.round(5).to_string(index=False))
    result.to_csv(f"{PROJECT_NAME}_tier1_sensitivity.csv", index=False)

/tmp/ipykernel_658/3992553079.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.rename(columns={'entrophy': 'entropy'}, inplace=True)


Loaded VSCode: 51846 rows



treatment  raw_ATE   raw_p  log1p_ATE  log1p_p  standardized_ATE_per_SD  standardized_p       Q25        Q75  Q25_to_Q75_contrast_days  sign_consistent_raw_vs_log  sig_at_05_raw  sig_at_05_log  sig_at_05_standardized
     ndev -0.06154 0.00000   -1.88078  0.00000                 -1.68660         0.00000  41.00000   87.00000                  -2.83068                        True           True           True                    True
  entropy  1.01923 0.00000    2.35937  0.00000                  1.04059         0.00000   0.00000    1.19470                   1.21768                        True           True           True                    True
     sexp -0.00018 0.00000   -0.51210  0.00000                 -1.04227         0.00000 366.00000 7273.75000                  -1.21163                        True           True           True                    True
     rexp -0.00046 0.17764   -0.30083  0.00086                 -0.19300         0.17764   1.06297    7.37824                  -0.002

treatment  raw_ATE   raw_p  log1p_ATE  log1p_p  standardized_ATE_per_SD  standardized_p       Q25        Q75  Q25_to_Q75_contrast_days  sign_consistent_raw_vs_log  sig_at_05_raw  sig_at_05_log  sig_at_05_standardized
     ndev -0.06154 0.00000   -1.88078  0.00000                 -1.68660         0.00000  41.00000   87.00000                  -2.83068                        True           True           True                    True
  entropy  1.01923 0.00000    2.35937  0.00000                  1.04059         0.00000   0.00000    1.19470                   1.21768                        True           True           True                    True
     sexp -0.00018 0.00000   -0.51210  0.00000                 -1.04227         0.00000 366.00000 7273.75000                  -1.21163                        True           True           True                    True
     rexp -0.00046 0.17764   -0.30083  0.00086                 -0.19300         0.17764   1.06297    7.37824                  -0.00288                        True          False           True                   False

In [8]:
print(result)

  treatment   raw_ATE         raw_p  log1p_ATE       log1p_p  standardized_ATE_per_SD  standardized_p         Q25          Q75  Q25_to_Q75_contrast_days  \
0      ndev -0.061536  4.488045e-32  -1.880784  4.504399e-24                -1.686604    4.488045e-32   41.000000    87.000000                 -2.830676   
1   entropy  1.019233  3.595333e-13   2.359371  8.926482e-16                 1.040586    3.595333e-13    0.000000     1.194704                  1.217682   
2      sexp -0.000175  3.295072e-13  -0.512102  8.838681e-18                -1.042270    3.295072e-13  366.000000  7273.750000                 -1.211634   
3      rexp -0.000456  1.776371e-01  -0.300826  8.560264e-04                -0.193002    1.776371e-01    1.062971     7.378240                 -0.002882   

   sign_consistent_raw_vs_log  sig_at_05_raw  sig_at_05_log  sig_at_05_standardized  
0                        True           True           True                    True  
1                        True           True   